# DDIM→DDIM inference on handscan validation data

Run reconstruction on the **healthy** (`clean plane 1`) and **unhealthy** (`streaks on plane 1`) crops prepared by `PrepareHandscanValidation.ipynb`.

**Model:** `/exp/sbnd/data/users/gputnam/training-SBND/iterE/results/brats2update111000.pt`  
**T:** 100  
**Procedure:** same DDIM-forward → DDIM-inverse recipe as `run_ddim2ddim_inference.py` / `AnalyzeDDIM2DDIM_Outputs.ipynb`.

For each input NPZ this writes:
- `*_T{T}_ddim2ddim.pkl` — per-patch `original`, `ddim2ddim-T{T}`, `saliency-T{T}`
- PNG side-by-side plots under `inference_T100/plots/{label}/`

**Kernel:** `env`. On CPU-only hosts expect ~3 min/patch (6 patches/file × 42 files ≈ 13 h). Prefer a GPU node when available.

You can also run the CLI wrapper from a terminal::

```bash
cd /exp/sbnd/app/users/munjung/anomaly-detection
PYTHONPATH=_stubs:train/diffusion-anomaly \
  /exp/sbnd/app/users/munjung/env/bin/python run_handscan_validation_inference.py \
    --T 100 --batch-size 1
```

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch as th
from tqdm.auto import tqdm

# Visdom stub + guided_diffusion on path
_REPO = Path(".").resolve()
sys.path.insert(0, str(_REPO / "_stubs"))
sys.path.insert(0, str(_REPO / "train" / "diffusion-anomaly"))

from guided_diffusion import dist_util
from run_ddim2ddim_inference import (
    build_model_and_diffusion,
    ddim2ddim_reconstruct,
    gather_patches_from_npz,
    visualize_np,
)
from run_handscan_validation_inference import run_one

# ── Configure ────────────────────────────────────────────────────────────────
MODEL_PATH = Path("/exp/sbnd/data/users/gputnam/training-SBND/iterE/results/brats2update111000.pt")
INPUT_ROOT = Path("handscan_validation/npz_inference")
OUTPUT_ROOT = Path("handscan_validation/inference_T100")
T = 100
BATCH_SIZE = 1
MAX_PATCHES = None          # set to 1 for a quick smoke test
MAX_FILES_PER_LABEL = None  # set to an int to limit files
LABELS = ["healthy", "unhealthy"]

plt.rcParams["figure.dpi"] = 110
print("Device:", dist_util.dev())
print("Model :", MODEL_PATH)
print("Input :", INPUT_ROOT.resolve())
print("Output:", OUTPUT_ROOT.resolve())

## Load model

In [ ]:
th.set_grad_enabled(False)

model, diffusion = build_model_and_diffusion()
sd = dist_util.load_state_dict(str(MODEL_PATH), map_location="cpu")
model.load_state_dict(sd)
model.to(dist_util.dev())
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {MODEL_PATH.name}")
print(f"Parameters: {n_params:,}")

## Run inference on healthy and unhealthy sets

Each file yields a pickle + one PNG per patch (`original | reconstructed | difference`).

In [ ]:
written: list[Path] = []

for label in LABELS:
    in_dir = INPUT_ROOT / label
    out_dir = OUTPUT_ROOT / label
    plot_dir = OUTPUT_ROOT / "plots" / label
    npzs = sorted(in_dir.glob("*.npz"))
    if MAX_FILES_PER_LABEL is not None:
        npzs = npzs[:MAX_FILES_PER_LABEL]
    print(f"\n=== {label}: {len(npzs)} files ===")
    for nz in tqdm(npzs, desc=label):
        pkl = run_one(
            nz,
            out_dir,
            plot_dir,
            diffusion,
            model,
            T=T,
            batch_size=BATCH_SIZE,
            max_patches=MAX_PATCHES,
            label=label,
        )
        written.append(pkl)

print(f"\nWrote {len(written)} pickle bundles under {OUTPUT_ROOT.resolve()}")
print("Plots under", (OUTPUT_ROOT / "plots").resolve())

## Spot-check one healthy and one unhealthy reconstruction

In [ ]:
import pickle


def load_first_patch(pkl_path: Path, T: int = T):
    with pkl_path.open("rb") as fh:
        raw = pickle.load(fh)
    keys = sorted(k for k in raw if isinstance(k, int))
    p = raw[keys[0]]
    return (
        np.squeeze(p["original"]),
        np.squeeze(p[f"ddim2ddim-T{T}"]),
        np.squeeze(p[f"saliency-T{T}"]),
        pkl_path.name,
    )


for label in LABELS:
    pkls = sorted((OUTPUT_ROOT / label).glob(f"*_T{T}_ddim2ddim.pkl"))
    if not pkls:
        print(f"No pickles yet for {label}")
        continue
    orig, reco, sal, name = load_first_patch(pkls[0])
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, img, title, v0, v1 in [
        (axes[0], orig, "Original", -1, 1),
        (axes[1], reco, "Reconstructed", -1, 1),
        (axes[2], sal, "Difference", -0.5, 0.5),
    ]:
        im = ax.imshow(img, aspect="auto", origin="lower", cmap="bwr", vmin=v0, vmax=v1)
        ax.set_title(title)
        fig.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle(f"{label}: {name}")
    fig.tight_layout()
    plt.show()